# MIB Crystal Analysis Quick Workflow

This notebook is a lightweight entry point for large MIB data. Keep reusable logic in `general_py4DSTEM_crystal_workflow.py` and the APP services; use this notebook for parameter selection, visual checks, and staged execution.

Default target: `DataCube (512, 512, 256, 256)`, about 32 GB. Always start with `MEMMAP` and a centered `128 x 128` ROI before full-dataset phase/orientation matching.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import py4DSTEM

MIB_FILE = Path(r'D:/data/sample.mib')
SCAN_SHAPE = (512, 512)
MEM_MODE = 'MEMMAP'
ROI = (192, 320, 192, 320)  # rx0, rx1, ry0, ry1; 128 x 128 centered tuning ROI
OUTPUT_DIR = Path('mib_crystal_output')
OUTPUT_DIR.mkdir(exist_ok=True)


## 1. Import and Mean Diffraction Pattern

MIB input must use `scan=(512, 512)` and `mem='MEMMAP'` unless you intentionally override them.


In [ ]:
dc = py4DSTEM.import_file(str(MIB_FILE), mem=MEM_MODE, scan=SCAN_SHAPE)
mean_dp = dc.get_dp_mean().data

plt.figure(figsize=(5, 5))
plt.imshow(np.log1p(mean_dp), cmap='gray')
plt.title('Mean diffraction pattern')
plt.axis('off');


## 2. ROI-First Bragg Detection

Tune Bragg-disk parameters on the ROI. Only switch to full dataset after the BVM and confidence maps are stable.


In [ ]:
rx0, rx1, ry0, ry1 = ROI
dc_roi = dc[rx0:rx1, ry0:ry1]

bragg_peaks = py4DSTEM.process.diffraction.find_Bragg_disks(
    dc_roi,
    corrPower=1.0,
    sigma=1,
    edgeBoundary=20,
    minRelativeIntensity=0.05,
    minPeakSpacing=8,
)

bvm = bragg_peaks.histogram(mode='cal')
plt.figure(figsize=(5, 5))
plt.imshow(np.log1p(bvm), cmap='magma')
plt.title('ROI Bragg Vector Map')
plt.axis('off');


## 3. Crystal Analysis

Phase matching and orientation matching share the same calibrated `BraggVectors`. Add candidate CIF files, build one orientation library per phase, then assign the phase with the highest score at each scan point.


In [ ]:
# Replace these with your actual CIF paths. v1 does not bundle a crystal database.
PHASES = {
    'Ti-fcc': Path(r'D:/data/Ti-fcc.cif'),
    'Ti-hcp': Path(r'D:/data/Ti-hcp.cif'),
}

crystals = {}
for phase_name, cif_path in PHASES.items():
    crystal = py4DSTEM.process.diffraction.Crystal.from_CIF(str(cif_path))
    crystal.setup_diffraction(accelerating_voltage=300_000)
    crystal.calculate_structure_factors(k_max=1.5, tol_structure_factor=1e-4)
    crystal.orientation_plan(
        zone_axis_range='auto',
        angle_step_zone_axis=5,
        angle_step_in_plane=5,
        progress_bar=True,
    )
    crystals[phase_name] = crystal


In [ ]:
orientation_maps = {}
score_maps = []
phase_names = list(crystals)

for phase_name, crystal in crystals.items():
    omap = crystal.match_orientations(
        bragg_peaks,
        num_matches_return=1,
        min_angle_between_matches_deg=2,
        min_number_peaks=3,
        progress_bar=True,
    )
    orientation_maps[phase_name] = omap
    score = np.asarray(getattr(omap, 'corr', getattr(omap, 'corr_map', None)), dtype=float)
    score_maps.append(score[:, :, 0] if score.ndim >= 3 else score)

score_stack = np.stack(score_maps, axis=0)
phase_id_map = np.argmax(score_stack, axis=0)
best_score = np.max(score_stack, axis=0)
sorted_score = np.sort(score_stack, axis=0)
confidence_gap = sorted_score[-1] - sorted_score[-2] if len(score_maps) > 1 else np.ones_like(best_score)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(phase_id_map, cmap='tab10')
axes[0].set_title('Phase ID')
axes[1].imshow(best_score, cmap='viridis')
axes[1].set_title('Best score')
axes[2].imshow(confidence_gap, cmap='inferno')
axes[2].set_title('Confidence gap')
for ax in axes: ax.axis('off')


## 4. Full Dataset

After ROI validation, repeat Bragg detection on `dc` instead of `dc_roi` and reuse the same CIF/orientation parameters. Grain and strain analysis are optional; if the installed py4DSTEM version lacks those APIs, keep phase/orientation results and record the warning.
